# Notebook A · Mencari *brute force* di log login
**Matrikulasi MTI UNLA · Sesi 1 · Versi mahasiswa**

Anda adalah analis keamanan. Pagi ini sistem login kampus terasa lambat. Tugas Anda: membaca log login dan menemukan apakah ada serangan.

**Cara mengerjakan**
1. Jalankan sel dari atas ke bawah (klik sel, lalu tekan **Shift + Enter**).
2. Di beberapa sel ada tanda `___`. Ganti tanda itu dengan jawaban Anda, lalu jalankan sel tersebut.
3. Tidak perlu instalasi apa pun; semua berjalan di Google Colab.
4. Kalau muncul pesan error yang menyebut `___`, berarti ada bagian yang belum Anda isi.

Data di notebook ini **sintetis** (dibuat untuk latihan). Alamat IP memakai blok dokumentasi resmi (192.0.2.x, 198.51.100.x, 203.0.113.x) yang tidak dimiliki siapa pun.

## Langkah 0 · Siapkan data
Sel di bawah membuat file `log_login.csv` berisi catatan login satu pagi. Cukup jalankan; Anda belum perlu memahami isinya.

In [29]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
tanggal = "2026-10-12"
baris = []

# Pengguna normal: login dari alamat yang biasa mereka pakai, sesekali salah ketik password
pengguna = ["sari", "budi", "dewi", "rizky", "putri", "agus", "lina", "fajar", "nina", "yoga",
            "hendra", "maya", "tono", "wulan", "dimas", "rani", "eko", "sinta", "bayu", "intan"]
ip_normal = [f"192.0.2.{i}" for i in range(10, 30)] + [f"203.0.113.{i}" for i in range(40, 50)]
for _ in range(620):
    u = rng.choice(pengguna)
    ip = ip_normal[pengguna.index(u) % len(ip_normal)] if rng.random() < 0.85 else rng.choice(ip_normal)
    detik = int(rng.integers(0, 4 * 3600))          # antara 08:00 dan 12:00
    status = "BERHASIL" if rng.random() > 0.08 else "GAGAL"
    baris.append((pd.Timestamp(f"{tanggal} 08:00:00") + pd.Timedelta(seconds=detik), ip, u, status))

# Baris yang tampil di slide
baris.append((pd.Timestamp(f"{tanggal} 08:02:40"), "192.0.2.10", "sari", "BERHASIL"))

# Penyerang: 200 percobaan gagal dalam 5 menit ke banyak username, lalu satu berhasil
penyerang = "198.51.100.24"
tebakan = ["admin", "admin", "root", "sari"] + list(rng.choice(
    ["admin", "root", "administrator", "operator", "test", "user", "sari", "budi", "dewi", "guest"], 196))
mulai = pd.Timestamp(f"{tanggal} 08:01:13")
jeda = np.sort(rng.integers(0, 299, 200)); jeda[:4] = [0, 1, 1, 2]
for t, u in zip(jeda, tebakan):
    baris.append((mulai + pd.Timedelta(seconds=int(t)), penyerang, u, "GAGAL"))
baris.append((pd.Timestamp(f"{tanggal} 08:06:31"), penyerang, "operator", "BERHASIL"))

log = pd.DataFrame(baris, columns=["waktu", "ip_sumber", "username", "status"]).sort_values("waktu", kind="stable")
log.to_csv("log_login.csv", index=False)
print("File log_login.csv dibuat:", len(log), "baris")

File log_login.csv dibuat: 822 baris


## Langkah 1 · Baca log
Setiap baris menjawab empat pertanyaan: **kapan**, **dari mana**, **siapa**, dan **berhasil atau tidak**.

In [36]:
log = pd.read_csv("log_login.csv", parse_dates=["waktu"])
log.head(10)

,waktu,ip_sumber,username,status
0,2026-10-12 08:01:13,198.51.100.24,admin,GAGAL
1,2026-10-12 08:01:14,192.0.2.13,rizky,GAGAL
2,2026-10-12 08:01:14,198.51.100.24,admin,GAGAL
3,2026-10-12 08:01:14,198.51.100.24,root,GAGAL
4,2026-10-12 08:01:15,198.51.100.24,sari,GAGAL
5,2026-10-12 08:01:18,198.51.100.24,sari,GAGAL
6,2026-10-12 08:01:19,198.51.100.24,test,GAGAL
7,2026-10-12 08:01:21,198.51.100.24,user,GAGAL
8,2026-10-12 08:01:24,198.51.100.24,admin,GAGAL
9,2026-10-12 08:01:26,198.51.100.24,test,GAGAL


In [37]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=log)

https://docs.google.com/spreadsheets/d/1eb4hvuE4OOR44TsIKBdu8kIHl_uXx9mCgiGs25BKB6E/edit#gid=0


## Langkah 2 · Berapa banyak login yang gagal?
Isi "status" dengan nama kolom yang berisi status login.

In [40]:
log["status"].value_counts()

,count
status,
BERHASIL,571
GAGAL,251


## Langkah 3 · Login gagal per alamat IP
Pengguna biasa sesekali salah ketik password. Satu alamat yang gagal ratusan kali patut dicurigai.

Isi "gagal" dengan nilai status untuk login yang gagal.

In [46]:
gagal = log[log["status"] == "GAGAL"]
gagal_per_ip = gagal.groupby("ip_sumber").size().sort_values(ascending=False)
gagal_per_ip.head(5)

,0
ip_sumber,
198.51.100.24,200
192.0.2.14,7
192.0.2.25,6
192.0.2.17,4
192.0.2.19,4


## Langkah 4 · Perilaku alamat yang paling mencurigakan
Isi "198.51.100.24" dengan alamat IP yang paling banyak gagal (lihat hasil Langkah 3, tulis di antara tanda kutip).

In [50]:
tersangka = "198.51.100.24"
aktivitas = log[log["ip_sumber"] == tersangka]

gagal_tersangka = aktivitas[aktivitas["status"] == "GAGAL"]
print("Jumlah login gagal      :", len(gagal_tersangka))
print("Pertama gagal           :", gagal_tersangka["waktu"].min())
print("Terakhir gagal          :", gagal_tersangka["waktu"].max())
print("Lama serangan           :", gagal_tersangka["waktu"].max() - gagal_tersangka["waktu"].min())
print("Username yang dicoba    :", gagal_tersangka["username"].nunique(), "berbeda")
print()
print("Apakah pernah berhasil? ")
aktivitas[aktivitas["status"] == "BERHASIL"]

Jumlah login gagal      : 200
Pertama gagal           : 2026-10-12 08:01:13
Terakhir gagal          : 2026-10-12 08:06:10
Lama serangan           : 0 days 00:04:57
Username yang dicoba    : 10 berbeda

Apakah pernah berhasil? 


,waktu,ip_sumber,username,status
217,2026-10-12 08:06:31,198.51.100.24,operator,BERHASIL


### Pertanyaan untuk Anda
1. Berapa kali alamat itu gagal login, dan dalam berapa menit?
   Alamat IP 198.51.100.24 mengalami 200 kali login gagal dalam waktu 4 menit 57 detik (sekitar 5 menit).


2. Apakah ia mencoba satu username atau banyak? Apa artinya?
   Alamat tersebut mencoba 10 username berbeda. Ini menunjukkan dugaan serangan brute force untuk menebak password dan mendapatkan akses ke sistem.


3. Lihat baris terakhir: alamat itu akhirnya **berhasil** masuk sebagai siapa? Apa yang harus segera dilakukan tim keamanan?
   Alamat tersebut berhasil masuk sebagai operator pada pukul 08:06:31. Tim keamanan harus segera mengamankan akun operator, mengganti password, mencabut sesi mencurigakan, memblokir IP penyerang, dan memeriksa log aktivitas.

*Tulis jawaban Anda di sel teks ini (klik dua kali untuk mengedit).*

## Langkah 5 · Buat aturan deteksi sederhana
Analis tidak membaca log satu per satu. Mereka menulis **aturan**: "beri peringatan jika satu IP gagal login **20 kali atau lebih** dalam **5 menit**."

Isi 20 dengan angka ambang batas dari aturan di atas.

In [54]:
ambang = 20

per_5_menit = (gagal.set_index("waktu")
                    .groupby("ip_sumber")
                    .resample("5min")
                    .size()
                    .rename("jumlah_gagal")
                    .reset_index())
peringatan = per_5_menit[per_5_menit["jumlah_gagal"] >= ambang]
peringatan.sort_values("jumlah_gagal", ascending=False)

,ip_sumber,waktu,jumlah_gagal
384,198.51.100.24,2026-10-12 08:00:00,159
385,198.51.100.24,2026-10-12 08:05:00,41


**Renungkan:** kalau ambang diturunkan menjadi 2, apa yang terjadi? (Coba ganti angkanya dan jalankan ulang.) Peringatan yang terlalu sering untuk pengguna biasa yang salah ketik disebut **alarm palsu (false positive)**. Konsep ini akan muncul lagi di Sesi 2.

## Bonus · Hash, sidik jari data
Jalankan, lalu ubah satu huruf di kalimat kedua dan lihat apa yang terjadi.

In [59]:
import hashlib

for kalimat in ["Transfer Rp100.000 ke Budi", "Transfer Rp100.000 ke Buda"]:
    print(kalimat, "->", hashlib.sha256(kalimat.encode()).hexdigest())

Transfer Rp100.000 ke Budi -> 11a0e6e1238357c4a832411757767af46d61a66c4895d1f84822c48374c41306
Transfer Rp100.000 ke Buda -> bd6742adffe431495efebde382b5cb45342aef74a68af7dad1bf82d025100ac4


---
**Selesai.** Yang baru Anda lakukan adalah pekerjaan dasar analis SOC: membaca log, mencari pola, menulis aturan deteksi. Di MK05 (Big Data & Security Analytics) hal yang sama dilakukan pada jutaan baris log.